# Example queries: `utility` (resstock_oedi)

Auto-generated from `tests/query_snapshots/utility.json`. Each cell
runs one entry from the snapshot suite. Regenerate by running the
matching test with `--update-snapshot` or `--overwrite-snapshot`.


In [1]:
from pathlib import Path
from buildstock_query import BuildStockQuery
import pandas as pd


## Construct the BuildStockQuery object

`cache_folder` points at the snapshot test cache directory so this
notebook reuses parquets that the test suite has already downloaded
from Athena. Queries that are already cached return immediately;
anything new still hits Athena.


In [2]:
# This notebook lives in `tests/example_notebooks/`; the snapshot test
# cache is its sibling `tests/query_snapshots/resstock_oedi_cache/`. Resolve
# the path relative to the notebook directory (`_dh[0]` is set by
# IPython at kernel startup; falls back to CWD outside Jupyter).
_NB_DIR = Path(_dh[0] if "_dh" in globals() else ".").resolve()
_CACHE = (_NB_DIR / "../query_snapshots/resstock_oedi_cache").resolve()
bsq = BuildStockQuery(
    "rescore",
    "buildstock_sdr",
    "resstock_2024_amy2018_release_2",
    buildstock_type="resstock",
    db_schema="resstock_oedi_vu",
    skip_reports=True,
    cache_folder=str(_CACHE),
)


INFO:buildstock_query.query_core:Loading resstock_2024_amy2018_release_2 ...


INFO:botocore.tokens:Loading cached SSO token for nrel-sso


## `utility_unit_counts_by_eiaid_with_state_groupby`

utility.aggregate_unit_counts_by_eiaid restricted to two utilities, grouped by state. Mirrors basic_usage_oedi.ipynb cell 7. Schema-restricted: comstock TOMLs don't define map_eiaid_column.


In [3]:
result = bsq.utility.aggregate_unit_counts_by_eiaid(eiaid_list=['4110', '14328'], group_by=['state'])
result.head() if hasattr(result, 'head') else result


INFO:buildstock_query.utility_query:Aggregating unit counts by eiaid


,eiaid,state,sample_count,units_count
0,14328,CA,28730,3.417140e+06
1,4110,IL,15880,2.563967e+06


## `utility_annual_by_eiaid_all`

utility.aggregate_annual_by_eiaid with no eiaid_list — pins the all-EIAIDs branch (no restrict). Mirrors basic_usage_oedi.ipynb cell 8.


In [4]:
result = bsq.utility.aggregate_annual_by_eiaid(
    enduses=['out.site_energy.net.energy_consumption', 'out.site_energy.total.energy_consumption'],
)
result.head() if hasattr(result, 'head') else result


,eiaid,sample_count,units_count,site_energy.net.energy_consumption,site_energy.total.energy_consumption
0,10000,4227,261337.858727,9.099893e+09,9.105774e+09
1,10005,1801,312719.379108,1.051243e+10,1.051243e+10
2,10009,3206,16872.057129,3.126843e+08,3.148031e+08
3,10012,319,4577.457673,1.565442e+08,1.565442e+08
4,10019,630,18020.497538,6.343734e+08,6.343734e+08


## `utility_ts_by_eiaid`

utility.aggregate_ts_by_eiaid for one utility (eiaid=15466 = Xcel Energy Colorado, ~9.3k samples / ~1.3M units in CO — by far the largest CO utility). Pinned only after the _split_restrict 3-bucket fix that routes eiaid_weights.eiaid restricts to the outer WHERE instead of the bs JOIN ON-clause. State-restricted to keep Athena cost bounded — every TS-side join must be partition-filtered (CLAUDE.md cost guardrail). Eiaid was changed from 4110 (no CO coverage → empty result) to 15466 to exercise the bs_per_bldg eiaid-disaggregation path with actual data.


In [5]:
result = bsq.utility.aggregate_ts_by_eiaid(
    eiaid_list=['15466'],
    enduses=['out.site_energy.total.energy_consumption'],
    group_by=['state', 'timestamp'],
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


INFO:buildstock_query.utility_query:Will submit request for ['15466']


INFO:buildstock_query.utility_query:Submitting query for 15466


INFO:buildstock_query.query_core:{'submitted': 0, 'running': 0, 'pending': 1, 'completed': 0, 'failed': 0}


INFO:buildstock_query.query_core:Submitted queries[0] (CACHED)


INFO:buildstock_query.query_core:{'submitted': 1, 'running': 0, 'pending': 0, 'completed': 1, 'failed': 0}


INFO:buildstock_query.query_core:Batch query completed. 


INFO:buildstock_query.query_core:Got result from Query [0] (CACHED)


INFO:buildstock_query.query_core:Concatenating the results.


,eiaid,state,timestamp,sample_count,units_count,site_energy.total.energy_consumption,query_id
0,15466,CO,2018-07-26 02:30:00,9273,1.317657e+06,237637.142328,0
1,15466,CO,2018-07-26 02:45:00,9273,1.317657e+06,233354.940708,0
2,15466,CO,2018-07-26 03:00:00,9273,1.317657e+06,228111.095628,0
3,15466,CO,2018-07-26 03:15:00,9273,1.317657e+06,225265.038594,0
4,15466,CO,2018-07-26 03:30:00,9273,1.317657e+06,221044.176836,0


## `utility_get_eiaids_co`

utility.get_eiaids restricted to CO — returns the list of utility EIAIDs serving any building in the state. Pins the bs ⋈ eiaid_weights JOIN shape with weighted units_count aggregation. Schema-restricted: comstock TOMLs don't define map_eiaid_column. The get_query_only path bypasses the in-memory cache so the SQL is always emitted.


In [6]:
result = bsq.utility.get_eiaids(restrict=[('state', ['CO'])])
result.head() if hasattr(result, 'head') else result


KeyError: 'eiaid'

## `utility_buildings_by_eiaids`

utility.get_buildings_by_eiaids for one utility. Pins the SELECT DISTINCT bldg_id JOIN eiaid_weights shape — used as a building-set lookup before downstream queries. Schema-restricted: comstock TOMLs don't define map_eiaid_column.


In [7]:
result = bsq.utility.get_buildings_by_eiaids(eiaids=['4110'])
result.head() if hasattr(result, 'head') else result


,bldg_id
0,435
1,1308
2,4136
3,10354
4,10867


## `utility_locations_by_eiaids`

utility.get_locations_by_eiaids for one utility. Pins the SELECT DISTINCT gisjoin FROM eiaid_weights shape — the simplest of the eiaid lookup family, returns a list of geographies served by the listed utilities.


In [8]:
result = bsq.utility.get_locations_by_eiaids(eiaids=['4110'])
result.head() if hasattr(result, 'head') else result


KeyError: 'gisjoin'

## `utility_calculate_tou_bill_flat_rate_monthly_co`

utility.calculate_tou_bill with a flat-rate (10c/kWh everywhere) rate map, restricted to CO (cost guardrail per CLAUDE.md — TS queries without state restrict scan ~3.2 TB on OEDI). Pins the TOU SQL shape (timestamp→month/weekend/hour MappedColumn rate, multiplied into a meter column) without depending on the example-CSV rate files. The rate_map is built from the JSON marker `rate_map_flat: 0.10` in the specialized test function. The meter_col uses an OEDI-style TS column; the method's default (`fuel_use__electricity__total__kwh`) is non-OEDI.


In [9]:
result = bsq.utility.calculate_tou_bill(
    meter_col='out.electricity.total.energy_consumption',
    group_by=['state'],
    restrict=[('state', ['CO'])],
    rate_map={(1, 0, 0): 0.1, (1, 0, 1): 0.1, (1, 0, 2): 0.1, (1, 0, 3): 0.1, (1, 0, 4): 0.1, (1, 0, 5): 0.1, (1, 0, 6): 0.1, (1, 0, 7): 0.1, (1, 0, 8): 0.1, (1, 0, 9): 0.1, (1, 0, 10): 0.1, (1, 0, 11): 0.1, (1, 0, 12): 0.1, (1, 0, 13): 0.1, (1, 0, 14): 0.1, (1, 0, 15): 0.1, (1, 0, 16): 0.1, (1, 0, 17): 0.1, (1, 0, 18): 0.1, (1, 0, 19): 0.1, (1, 0, 20): 0.1, (1, 0, 21): 0.1, (1, 0, 22): 0.1, (1, 0, 23): 0.1, (1, 1, 0): 0.1, (1, 1, 1): 0.1, (1, 1, 2): 0.1, (1, 1, 3): 0.1, (1, 1, 4): 0.1, (1, 1, 5): 0.1, (1, 1, 6): 0.1, (1, 1, 7): 0.1, (1, 1, 8): 0.1, (1, 1, 9): 0.1, (1, 1, 10): 0.1, (1, 1, 11): 0.1, (1, 1, 12): 0.1, (1, 1, 13): 0.1, (1, 1, 14): 0.1, (1, 1, 15): 0.1, (1, 1, 16): 0.1, (1, 1, 17): 0.1, (1, 1, 18): 0.1, (1, 1, 19): 0.1, (1, 1, 20): 0.1, (1, 1, 21): 0.1, (1, 1, 22): 0.1, (1, 1, 23): 0.1, (2, 0, 0): 0.1, (2, 0, 1): 0.1, (2, 0, 2): 0.1, (2, 0, 3): 0.1, (2, 0, 4): 0.1, (2, 0, 5): 0.1, (2, 0, 6): 0.1, (2, 0, 7): 0.1, (2, 0, 8): 0.1, (2, 0, 9): 0.1, (2, 0, 10): 0.1, (2, 0, 11): 0.1, (2, 0, 12): 0.1, (2, 0, 13): 0.1, (2, 0, 14): 0.1, (2, 0, 15): 0.1, (2, 0, 16): 0.1, (2, 0, 17): 0.1, (2, 0, 18): 0.1, (2, 0, 19): 0.1, (2, 0, 20): 0.1, (2, 0, 21): 0.1, (2, 0, 22): 0.1, (2, 0, 23): 0.1, (2, 1, 0): 0.1, (2, 1, 1): 0.1, (2, 1, 2): 0.1, (2, 1, 3): 0.1, (2, 1, 4): 0.1, (2, 1, 5): 0.1, (2, 1, 6): 0.1, (2, 1, 7): 0.1, (2, 1, 8): 0.1, (2, 1, 9): 0.1, (2, 1, 10): 0.1, (2, 1, 11): 0.1, (2, 1, 12): 0.1, (2, 1, 13): 0.1, (2, 1, 14): 0.1, (2, 1, 15): 0.1, (2, 1, 16): 0.1, (2, 1, 17): 0.1, (2, 1, 18): 0.1, (2, 1, 19): 0.1, (2, 1, 20): 0.1, (2, 1, 21): 0.1, (2, 1, 22): 0.1, (2, 1, 23): 0.1, (3, 0, 0): 0.1, (3, 0, 1): 0.1, (3, 0, 2): 0.1, (3, 0, 3): 0.1, (3, 0, 4): 0.1, (3, 0, 5): 0.1, (3, 0, 6): 0.1, (3, 0, 7): 0.1, (3, 0, 8): 0.1, (3, 0, 9): 0.1, (3, 0, 10): 0.1, (3, 0, 11): 0.1, (3, 0, 12): 0.1, (3, 0, 13): 0.1, (3, 0, 14): 0.1, (3, 0, 15): 0.1, (3, 0, 16): 0.1, (3, 0, 17): 0.1, (3, 0, 18): 0.1, (3, 0, 19): 0.1, (3, 0, 20): 0.1, (3, 0, 21): 0.1, (3, 0, 22): 0.1, (3, 0, 23): 0.1, (3, 1, 0): 0.1, (3, 1, 1): 0.1, (3, 1, 2): 0.1, (3, 1, 3): 0.1, (3, 1, 4): 0.1, (3, 1, 5): 0.1, (3, 1, 6): 0.1, (3, 1, 7): 0.1, (3, 1, 8): 0.1, (3, 1, 9): 0.1, (3, 1, 10): 0.1, (3, 1, 11): 0.1, (3, 1, 12): 0.1, (3, 1, 13): 0.1, (3, 1, 14): 0.1, (3, 1, 15): 0.1, (3, 1, 16): 0.1, (3, 1, 17): 0.1, (3, 1, 18): 0.1, (3, 1, 19): 0.1, (3, 1, 20): 0.1, (3, 1, 21): 0.1, (3, 1, 22): 0.1, (3, 1, 23): 0.1, (4, 0, 0): 0.1, (4, 0, 1): 0.1, (4, 0, 2): 0.1, (4, 0, 3): 0.1, (4, 0, 4): 0.1, (4, 0, 5): 0.1, (4, 0, 6): 0.1, (4, 0, 7): 0.1, (4, 0, 8): 0.1, (4, 0, 9): 0.1, (4, 0, 10): 0.1, (4, 0, 11): 0.1, (4, 0, 12): 0.1, (4, 0, 13): 0.1, (4, 0, 14): 0.1, (4, 0, 15): 0.1, (4, 0, 16): 0.1, (4, 0, 17): 0.1, (4, 0, 18): 0.1, (4, 0, 19): 0.1, (4, 0, 20): 0.1, (4, 0, 21): 0.1, (4, 0, 22): 0.1, (4, 0, 23): 0.1, (4, 1, 0): 0.1, (4, 1, 1): 0.1, (4, 1, 2): 0.1, (4, 1, 3): 0.1, (4, 1, 4): 0.1, (4, 1, 5): 0.1, (4, 1, 6): 0.1, (4, 1, 7): 0.1, (4, 1, 8): 0.1, (4, 1, 9): 0.1, (4, 1, 10): 0.1, (4, 1, 11): 0.1, (4, 1, 12): 0.1, (4, 1, 13): 0.1, (4, 1, 14): 0.1, (4, 1, 15): 0.1, (4, 1, 16): 0.1, (4, 1, 17): 0.1, (4, 1, 18): 0.1, (4, 1, 19): 0.1, (4, 1, 20): 0.1, (4, 1, 21): 0.1, (4, 1, 22): 0.1, (4, 1, 23): 0.1, (5, 0, 0): 0.1, (5, 0, 1): 0.1, (5, 0, 2): 0.1, (5, 0, 3): 0.1, (5, 0, 4): 0.1, (5, 0, 5): 0.1, (5, 0, 6): 0.1, (5, 0, 7): 0.1, (5, 0, 8): 0.1, (5, 0, 9): 0.1, (5, 0, 10): 0.1, (5, 0, 11): 0.1, (5, 0, 12): 0.1, (5, 0, 13): 0.1, (5, 0, 14): 0.1, (5, 0, 15): 0.1, (5, 0, 16): 0.1, (5, 0, 17): 0.1, (5, 0, 18): 0.1, (5, 0, 19): 0.1, (5, 0, 20): 0.1, (5, 0, 21): 0.1, (5, 0, 22): 0.1, (5, 0, 23): 0.1, (5, 1, 0): 0.1, (5, 1, 1): 0.1, (5, 1, 2): 0.1, (5, 1, 3): 0.1, (5, 1, 4): 0.1, (5, 1, 5): 0.1, (5, 1, 6): 0.1, (5, 1, 7): 0.1, (5, 1, 8): 0.1, (5, 1, 9): 0.1, (5, 1, 10): 0.1, (5, 1, 11): 0.1, (5, 1, 12): 0.1, (5, 1, 13): 0.1, (5, 1, 14): 0.1, (5, 1, 15): 0.1, (5, 1, 16): 0.1, (5, 1, 17): 0.1, (5, 1, 18): 0.1, (5, 1, 19): 0.1, (5, 1, 20): 0.1, (5, 1, 21): 0.1, (5, 1, 22): 0.1, (5, 1, 23): 0.1, (6, 0, 0): 0.1, (6, 0, 1): 0.1, (6, 0, 2): 0.1, (6, 0, 3): 0.1, (6, 0, 4): 0.1, (6, 0, 5): 0.1, (6, 0, 6): 0.1, (6, 0, 7): 0.1, (6, 0, 8): 0.1, (6, 0, 9): 0.1, (6, 0, 10): 0.1, (6, 0, 11): 0.1, (6, 0, 12): 0.1, (6, 0, 13): 0.1, (6, 0, 14): 0.1, (6, 0, 15): 0.1, (6, 0, 16): 0.1, (6, 0, 17): 0.1, (6, 0, 18): 0.1, (6, 0, 19): 0.1, (6, 0, 20): 0.1, (6, 0, 21): 0.1, (6, 0, 22): 0.1, (6, 0, 23): 0.1, (6, 1, 0): 0.1, (6, 1, 1): 0.1, (6, 1, 2): 0.1, (6, 1, 3): 0.1, (6, 1, 4): 0.1, (6, 1, 5): 0.1, (6, 1, 6): 0.1, (6, 1, 7): 0.1, (6, 1, 8): 0.1, (6, 1, 9): 0.1, (6, 1, 10): 0.1, (6, 1, 11): 0.1, (6, 1, 12): 0.1, (6, 1, 13): 0.1, (6, 1, 14): 0.1, (6, 1, 15): 0.1, (6, 1, 16): 0.1, (6, 1, 17): 0.1, (6, 1, 18): 0.1, (6, 1, 19): 0.1, (6, 1, 20): 0.1, (6, 1, 21): 0.1, (6, 1, 22): 0.1, (6, 1, 23): 0.1, (7, 0, 0): 0.1, (7, 0, 1): 0.1, (7, 0, 2): 0.1, (7, 0, 3): 0.1, (7, 0, 4): 0.1, (7, 0, 5): 0.1, (7, 0, 6): 0.1, (7, 0, 7): 0.1, (7, 0, 8): 0.1, (7, 0, 9): 0.1, (7, 0, 10): 0.1, (7, 0, 11): 0.1, (7, 0, 12): 0.1, (7, 0, 13): 0.1, (7, 0, 14): 0.1, (7, 0, 15): 0.1, (7, 0, 16): 0.1, (7, 0, 17): 0.1, (7, 0, 18): 0.1, (7, 0, 19): 0.1, (7, 0, 20): 0.1, (7, 0, 21): 0.1, (7, 0, 22): 0.1, (7, 0, 23): 0.1, (7, 1, 0): 0.1, (7, 1, 1): 0.1, (7, 1, 2): 0.1, (7, 1, 3): 0.1, (7, 1, 4): 0.1, (7, 1, 5): 0.1, (7, 1, 6): 0.1, (7, 1, 7): 0.1, (7, 1, 8): 0.1, (7, 1, 9): 0.1, (7, 1, 10): 0.1, (7, 1, 11): 0.1, (7, 1, 12): 0.1, (7, 1, 13): 0.1, (7, 1, 14): 0.1, (7, 1, 15): 0.1, (7, 1, 16): 0.1, (7, 1, 17): 0.1, (7, 1, 18): 0.1, (7, 1, 19): 0.1, (7, 1, 20): 0.1, (7, 1, 21): 0.1, (7, 1, 22): 0.1, (7, 1, 23): 0.1, (8, 0, 0): 0.1, (8, 0, 1): 0.1, (8, 0, 2): 0.1, (8, 0, 3): 0.1, (8, 0, 4): 0.1, (8, 0, 5): 0.1, (8, 0, 6): 0.1, (8, 0, 7): 0.1, (8, 0, 8): 0.1, (8, 0, 9): 0.1, (8, 0, 10): 0.1, (8, 0, 11): 0.1, (8, 0, 12): 0.1, (8, 0, 13): 0.1, (8, 0, 14): 0.1, (8, 0, 15): 0.1, (8, 0, 16): 0.1, (8, 0, 17): 0.1, (8, 0, 18): 0.1, (8, 0, 19): 0.1, (8, 0, 20): 0.1, (8, 0, 21): 0.1, (8, 0, 22): 0.1, (8, 0, 23): 0.1, (8, 1, 0): 0.1, (8, 1, 1): 0.1, (8, 1, 2): 0.1, (8, 1, 3): 0.1, (8, 1, 4): 0.1, (8, 1, 5): 0.1, (8, 1, 6): 0.1, (8, 1, 7): 0.1, (8, 1, 8): 0.1, (8, 1, 9): 0.1, (8, 1, 10): 0.1, (8, 1, 11): 0.1, (8, 1, 12): 0.1, (8, 1, 13): 0.1, (8, 1, 14): 0.1, (8, 1, 15): 0.1, (8, 1, 16): 0.1, (8, 1, 17): 0.1, (8, 1, 18): 0.1, (8, 1, 19): 0.1, (8, 1, 20): 0.1, (8, 1, 21): 0.1, (8, 1, 22): 0.1, (8, 1, 23): 0.1, (9, 0, 0): 0.1, (9, 0, 1): 0.1, (9, 0, 2): 0.1, (9, 0, 3): 0.1, (9, 0, 4): 0.1, (9, 0, 5): 0.1, (9, 0, 6): 0.1, (9, 0, 7): 0.1, (9, 0, 8): 0.1, (9, 0, 9): 0.1, (9, 0, 10): 0.1, (9, 0, 11): 0.1, (9, 0, 12): 0.1, (9, 0, 13): 0.1, (9, 0, 14): 0.1, (9, 0, 15): 0.1, (9, 0, 16): 0.1, (9, 0, 17): 0.1, (9, 0, 18): 0.1, (9, 0, 19): 0.1, (9, 0, 20): 0.1, (9, 0, 21): 0.1, (9, 0, 22): 0.1, (9, 0, 23): 0.1, (9, 1, 0): 0.1, (9, 1, 1): 0.1, (9, 1, 2): 0.1, (9, 1, 3): 0.1, (9, 1, 4): 0.1, (9, 1, 5): 0.1, (9, 1, 6): 0.1, (9, 1, 7): 0.1, (9, 1, 8): 0.1, (9, 1, 9): 0.1, (9, 1, 10): 0.1, (9, 1, 11): 0.1, (9, 1, 12): 0.1, (9, 1, 13): 0.1, (9, 1, 14): 0.1, (9, 1, 15): 0.1, (9, 1, 16): 0.1, (9, 1, 17): 0.1, (9, 1, 18): 0.1, (9, 1, 19): 0.1, (9, 1, 20): 0.1, (9, 1, 21): 0.1, (9, 1, 22): 0.1, (9, 1, 23): 0.1, (10, 0, 0): 0.1, (10, 0, 1): 0.1, (10, 0, 2): 0.1, (10, 0, 3): 0.1, (10, 0, 4): 0.1, (10, 0, 5): 0.1, (10, 0, 6): 0.1, (10, 0, 7): 0.1, (10, 0, 8): 0.1, (10, 0, 9): 0.1, (10, 0, 10): 0.1, (10, 0, 11): 0.1, (10, 0, 12): 0.1, (10, 0, 13): 0.1, (10, 0, 14): 0.1, (10, 0, 15): 0.1, (10, 0, 16): 0.1, (10, 0, 17): 0.1, (10, 0, 18): 0.1, (10, 0, 19): 0.1, (10, 0, 20): 0.1, (10, 0, 21): 0.1, (10, 0, 22): 0.1, (10, 0, 23): 0.1, (10, 1, 0): 0.1, (10, 1, 1): 0.1, (10, 1, 2): 0.1, (10, 1, 3): 0.1, (10, 1, 4): 0.1, (10, 1, 5): 0.1, (10, 1, 6): 0.1, (10, 1, 7): 0.1, (10, 1, 8): 0.1, (10, 1, 9): 0.1, (10, 1, 10): 0.1, (10, 1, 11): 0.1, (10, 1, 12): 0.1, (10, 1, 13): 0.1, (10, 1, 14): 0.1, (10, 1, 15): 0.1, (10, 1, 16): 0.1, (10, 1, 17): 0.1, (10, 1, 18): 0.1, (10, 1, 19): 0.1, (10, 1, 20): 0.1, (10, 1, 21): 0.1, (10, 1, 22): 0.1, (10, 1, 23): 0.1, (11, 0, 0): 0.1, (11, 0, 1): 0.1, (11, 0, 2): 0.1, (11, 0, 3): 0.1, (11, 0, 4): 0.1, (11, 0, 5): 0.1, (11, 0, 6): 0.1, (11, 0, 7): 0.1, (11, 0, 8): 0.1, (11, 0, 9): 0.1, (11, 0, 10): 0.1, (11, 0, 11): 0.1, (11, 0, 12): 0.1, (11, 0, 13): 0.1, (11, 0, 14): 0.1, (11, 0, 15): 0.1, (11, 0, 16): 0.1, (11, 0, 17): 0.1, (11, 0, 18): 0.1, (11, 0, 19): 0.1, (11, 0, 20): 0.1, (11, 0, 21): 0.1, (11, 0, 22): 0.1, (11, 0, 23): 0.1, (11, 1, 0): 0.1, (11, 1, 1): 0.1, (11, 1, 2): 0.1, (11, 1, 3): 0.1, (11, 1, 4): 0.1, (11, 1, 5): 0.1, (11, 1, 6): 0.1, (11, 1, 7): 0.1, (11, 1, 8): 0.1, (11, 1, 9): 0.1, (11, 1, 10): 0.1, (11, 1, 11): 0.1, (11, 1, 12): 0.1, (11, 1, 13): 0.1, (11, 1, 14): 0.1, (11, 1, 15): 0.1, (11, 1, 16): 0.1, (11, 1, 17): 0.1, (11, 1, 18): 0.1, (11, 1, 19): 0.1, (11, 1, 20): 0.1, (11, 1, 21): 0.1, (11, 1, 22): 0.1, (11, 1, 23): 0.1, (12, 0, 0): 0.1, (12, 0, 1): 0.1, (12, 0, 2): 0.1, (12, 0, 3): 0.1, (12, 0, 4): 0.1, (12, 0, 5): 0.1, (12, 0, 6): 0.1, (12, 0, 7): 0.1, (12, 0, 8): 0.1, (12, 0, 9): 0.1, (12, 0, 10): 0.1, (12, 0, 11): 0.1, (12, 0, 12): 0.1, (12, 0, 13): 0.1, (12, 0, 14): 0.1, (12, 0, 15): 0.1, (12, 0, 16): 0.1, (12, 0, 17): 0.1, (12, 0, 18): 0.1, (12, 0, 19): 0.1, (12, 0, 20): 0.1, (12, 0, 21): 0.1, (12, 0, 22): 0.1, (12, 0, 23): 0.1, (12, 1, 0): 0.1, (12, 1, 1): 0.1, (12, 1, 2): 0.1, (12, 1, 3): 0.1, (12, 1, 4): 0.1, (12, 1, 5): 0.1, (12, 1, 6): 0.1, (12, 1, 7): 0.1, (12, 1, 8): 0.1, (12, 1, 9): 0.1, (12, 1, 10): 0.1, (12, 1, 11): 0.1, (12, 1, 12): 0.1, (12, 1, 13): 0.1, (12, 1, 14): 0.1, (12, 1, 15): 0.1, (12, 1, 16): 0.1, (12, 1, 17): 0.1, (12, 1, 18): 0.1, (12, 1, 19): 0.1, (12, 1, 20): 0.1, (12, 1, 21): 0.1, (12, 1, 22): 0.1, (12, 1, 23): 0.1},
)
result.head() if hasattr(result, 'head') else result


,timestamp,state,sample_count,units_count,rows_per_sample,electricity.total.energy_consumption__tou__dollars
0,2018-01-01,CO,9425,2.377943e+06,2976,2.529477e+06
1,2018-02-01,CO,9425,2.377943e+06,2688,2.335684e+06
2,2018-03-01,CO,9425,2.377943e+06,2976,1.849398e+06
3,2018-04-01,CO,9425,2.377943e+06,2880,1.588177e+06
4,2018-05-01,CO,9425,2.377943e+06,2976,1.477754e+06
